# Stage 3 — PETSc: towards parallel and 3D

The SciPy direct solver costs O(N^1.5–2) and can't run across cluster nodes. In 3D that's fatal.
PETSc gives MPI-parallel iterative solvers. Step 3a: solve *the same matrix* we already trust, iteratively.

The Stokes system is a **saddle-point** problem [[K, G], [Gᵀ, 0]]: K acts on velocity, G is the pressure gradient.
Strategy (standard for Stokes): FGMRES on the whole system, preconditioned by a **Schur-complement split**:
- velocity block K → inner CG (later: geometric multigrid)
- Schur complement S = −Gᵀ K⁻¹ G ≈ −diag(1/η) (the pressure "mass matrix", which works because Stokes is well-posed)
- the arbitrary pressure constant → declared as a **null space** (no pinned cell)

The test that matters: **outer iterations must not grow with grid size or viscosity contrast.** That's what makes 3D feasible.

In [1]:
import time
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from petsc4py import PETSc
from ganymede.stokes import Grid, stokes_matrix_fast, solve_stokes, unpack
print("PETSc", ".".join(map(str, PETSc.Sys.getVersion())), "| DMStag available:", hasattr(PETSc, "DMStag"))

PETSc 3.22.2 | DMStag available: True


In [2]:
def stokes_symmetric(g, eta_c, eta_n):
    """The same staggered-grid discretisation, rearranged for iterative solvers:
      1. no pinned pressure: every cell keeps its continuity equation (the solver handles the constant via a null space)
      2. all rows multiplied by -1 -> velocity block symmetric positive definite, whole system [[K, G], [G^T, 0]]
      3. wall velocities (= 0) decoupled: their columns removed, their rows scaled like the interior
    Returns the matrix A, the row signs s (apply to the RHS too), and the wall-velocity rows."""
    A = stokes_matrix_fast(g, eta_c, eta_n).tocsr()
    nv = g.n_vx + g.n_vz
    # 1. restore the continuity equation of cell (0, 0)
    r0 = nv
    A = A.tolil()
    A.rows[r0], A.data[r0] = [], []
    for col, val in ((g.ivx(0, 1), 1 / g.dx), (g.ivx(0, 0), -1 / g.dx), (g.ivz(1, 0), 1 / g.dz), (g.ivz(0, 0), -1 / g.dz)):
        A[r0, col] = val
    A = A.tocsr()
    # 2. flip signs
    s = -np.ones(A.shape[0])
    A = sp.diags(s) @ A
    # 3. wall velocities: rows that are pure identity rows in the velocity part
    d = A.diagonal()
    nnz_row = np.diff(A.indptr)
    walls = np.where((nnz_row[:nv] == 1) & np.isclose(d[:nv], -1.0))[0]
    scale = np.median(np.abs(np.delete(d[:nv], walls)))
    keep = np.ones(A.shape[0]); keep[walls] = 0.0
    A = (A @ sp.diags(keep)).tocsr()
    A = A + sp.csr_matrix((np.full(len(walls), scale), (walls, walls)), shape=A.shape)
    A.eliminate_zeros()
    A = (A + sp.diags(np.zeros(A.shape[0]))).tocsr()     # explicit diagonal entries (PETSc wants them)
    return A, s, walls

In [3]:
def petsc_stokes_solve(g, eta_c, eta_n, rho_vz, rtol=1e-10, u_pc="ilu", u_rtol=1e-8, max_it=300):
    """FGMRES on the full system, preconditioned by a Schur-complement split:
       velocity block K      -> inner CG + u_pc ('ilu' here; geometric multigrid later)
       Schur complement S    -> approximated by -diag(1/eta)  (the classic Stokes 'pressure mass matrix')
       constant pressure     -> declared as a null space."""
    A, s, walls = stokes_symmetric(g, eta_c, eta_n)
    b = np.zeros(A.shape[0])
    bz = rho_vz.copy(); bz[0] = bz[-1] = 0.0
    b[g.n_vx:g.n_vx + g.n_vz] = bz.ravel()
    b *= s; b[walls] = 0.0
    nv = g.n_vx + g.n_vz

    M = PETSc.Mat().createAIJ(size=A.shape, csr=(A.indptr, A.indices, A.data))
    null = M.createVecRight(); null.setArray(np.r_[np.zeros(nv), np.full(g.n_p, 1 / np.sqrt(g.n_p))])
    M.setNullSpace(PETSc.NullSpace().create(vectors=[null]))
    S = sp.diags(-1.0 / eta_c.ravel()).tocsr()
    S_pc = PETSc.Mat().createAIJ(size=S.shape, csr=(S.indptr, S.indices, S.data))

    ksp = PETSc.KSP().create()
    ksp.setOperators(M); ksp.setType("fgmres")
    pc = ksp.getPC(); pc.setType("fieldsplit")
    pc.setFieldSplitIS(("u", PETSc.IS().createStride(nv, 0, 1)), ("p", PETSc.IS().createStride(g.n_p, nv, 1)))
    pc.setFieldSplitType(PETSc.PC.CompositeType.SCHUR)
    pc.setFieldSplitSchurFactType(PETSc.PC.FieldSplitSchurFactType.UPPER)
    pc.setFieldSplitSchurPreType(PETSc.PC.FieldSplitSchurPreType.USER, S_pc)
    opts = PETSc.Options()
    opts["fieldsplit_u_ksp_type"] = "cg";      opts["fieldsplit_u_pc_type"] = u_pc
    opts["fieldsplit_u_ksp_rtol"] = u_rtol
    opts["fieldsplit_p_ksp_type"] = "preonly"; opts["fieldsplit_p_pc_type"] = "jacobi"
    pc.setFromOptions()
    ksp.setTolerances(rtol=rtol, max_it=max_it)

    x, rhs = M.createVecRight(), M.createVecLeft()
    rhs.setArray(b)
    t0 = time.time(); ksp.solve(rhs, x); elapsed = time.time() - t0
    vx, vz, P = unpack(g, x.getArray().copy())
    return vx, vz, P, ksp.getIterationNumber(), ksp.getConvergedReason(), elapsed

In [4]:
k = np.pi
print("isoviscous, sinusoidal density (analytic solution)")
for n in (32, 64, 128, 256):
    g = Grid(n, n)
    Xz, Zz = np.meshgrid(g.xc, g.zn); Xx, Zx = np.meshgrid(g.xn, g.zc)
    vx, vz, P, its, reason, dt = petsc_stokes_solve(g, np.ones((n, n)), np.ones((n + 1, n + 1)), np.cos(k * Xz) * np.sin(k * Zz))
    A0 = k / (2 * k**2) ** 2
    err = max(np.abs(vx - A0 * k * np.sin(k * Xx) * np.cos(k * Zx)).max(),
              np.abs(vz + A0 * k * np.cos(k * Xz) * np.sin(k * Zz)).max()) / (A0 * k)
    print(f"  {n:4d}²: error {err:.2e}, outer iterations {its}, converged {reason > 0}, {dt:.2f} s")

print("\nstiff lid: eta = contrast^z, compared with the SciPy direct solver")
for contrast in (1e3, 1e6, 1e8):
    n = 128; g = Grid(n, n)
    Xz, Zz = np.meshgrid(g.xc, g.zn)
    eta_c = contrast ** np.meshgrid(g.xc, g.zc)[1]; eta_n = contrast ** np.meshgrid(g.xn, g.zn)[1]
    rho = np.cos(k * Xz) * np.sin(k * Zz)
    vx0, vz0, _ = solve_stokes(g, eta_c, eta_n, rho)
    vx, vz, P, its, reason, dt = petsc_stokes_solve(g, eta_c, eta_n, rho)
    diff = max(np.abs(vx - vx0).max(), np.abs(vz - vz0).max()) / np.abs(vz0).max()
    print(f"  contrast {contrast:.0e}: |PETSc − SciPy| / |v| = {diff:.1e}, outer iterations {its}, {dt:.2f} s")

isoviscous, sinusoidal density (analytic solution)
    32²: error 8.03e-04, outer iterations 4, converged True, 0.01 s
    64²: error 2.01e-04, outer iterations 3, converged True, 0.05 s
   128²: error 5.02e-05, outer iterations 3, converged True, 0.58 s
   256²: error 1.25e-05, outer iterations 3, converged True, 3.59 s

stiff lid: eta = contrast^z, compared with the SciPy direct solver
  contrast 1e+03: |PETSc − SciPy| / |v| = 8.3e-09, outer iterations 8, 1.32 s
  contrast 1e+06: |PETSc − SciPy| / |v| = 3.7e-08, outer iterations 9, 1.25 s
  contrast 1e+08: |PETSc − SciPy| / |v| = 6.1e-08, outer iterations 10, 1.16 s


## Stage 3b — DMStag and geometric multigrid

DMStag is PETSc's staggered-grid manager: the same layout we built by hand (P at centres, v on faces), but it knows
how to split the grid across MPI processes and how to **coarsen** it, which is what geometric multigrid needs.

Plan for this step: keep our assembly, but
1. map our unknown ordering [vx…, vz…, P…] onto DMStag's ordering (a pure permutation),
2. build a second, velocity-only DMStag; its coarsening hierarchy provides the multigrid interpolation,
3. multigrid on the velocity block, coarse operators by Galerkin projection (Rᵀ A P) of our fine matrix.

Target: inner iterations independent of grid size, cost ∝ N.

In [5]:
def make_dmstag(nx, nz, L=1.0, H=1.0, dofs=(0, 1, 1)):
    """2D DMStag; dofs = (per vertex, per face, per element). (0,1,1): v on faces, P in cells. (0,1,0): velocity only."""
    dm = PETSc.DMStag().create(dim=2, dofs=dofs, sizes=(nx, nz),
                               boundary_types=(PETSc.DM.BoundaryType.NONE,) * 2,
                               stencil_type=PETSc.DMStag.StencilType.BOX, stencil_width=1, setUp=True)
    dm.setUniformCoordinatesProduct(0.0, L, 0.0, H)
    return dm

def stag_index_array(dm):
    """[j, i, slot] -> DMStag global index (-1 where a slot doesn't exist, e.g. beyond the right/top boundary)."""
    v = dm.createGlobalVec(); v.setArray(np.arange(v.getSize(), dtype=float) + 1)
    loc = dm.createLocalVec(); dm.globalToLocal(v, loc)
    _, (gnx, gnz) = dm.getGhostCorners()
    return loc.getArray().reshape(gnz, gnx, -1).astype(int) - 1

SL = PETSc.DMStag.StencilLocation

def ours_to_stag(g, dm):
    """perm[k] = DMStag global index of our unknown k (ordering [vx..., vz..., P...])."""
    a = stag_index_array(dm)
    vx = a[:g.nz, :g.nx + 1, dm.getLocationSlot(SL.LEFT, 0)]      # vx[j,i]: left face of point (i,j)
    vz = a[:g.nz + 1, :g.nx, dm.getLocationSlot(SL.DOWN, 0)]      # vz[j,i]: bottom face
    P  = a[:g.nz, :g.nx, dm.getLocationSlot(SL.ELEMENT, 0)]
    perm = np.concatenate([vx.ravel(), vz.ravel(), P.ravel()])
    assert (perm >= 0).all() and np.unique(perm).size == perm.size == dm.createGlobalVec().getSize()
    return perm

def velocity_indices(g, dm):
    """Global indices of all velocity unknowns of a DMStag, in our [vx..., vz...] order."""
    a = stag_index_array(dm)
    return np.concatenate([a[:g.nz, :g.nx + 1, dm.getLocationSlot(SL.LEFT, 0)].ravel(),
                           a[:g.nz + 1, :g.nx, dm.getLocationSlot(SL.DOWN, 0)].ravel()])

# --- checks on a small, non-square grid ---
nx, nz = 16, 12
g = Grid(nx, nz); dm = make_dmstag(nx, nz); dm_v = make_dmstag(nx, nz, dofs=(0, 1, 0))
perm = ours_to_stag(g, dm)
A = stokes_matrix_fast(g, np.ones((nz, nx)), np.ones((nz + 1, nx + 1)))
Xz, Zz = np.meshgrid(g.xc, g.zn)
b = np.zeros(A.shape[0]); bz = np.cos(k * Xz) * np.sin(k * Zz); bz[0] = bz[-1] = 0; b[g.n_vx:g.n_vx + g.n_vz] = bz.ravel()
n = A.shape[0]
Pm = sp.csr_matrix((np.ones(n), (perm, np.arange(n))), shape=(n, n))
from scipy.sparse.linalg import spsolve
x_ours = spsolve(A, b); x_stag = spsolve((Pm @ A @ Pm.T).tocsc(), Pm @ b)
print("same solution in DMStag ordering:", np.abs(x_ours - x_stag[perm]).max())
order = np.argsort(velocity_indices(g, dm))
print("velocity block order == velocity-only DMStag order:", np.array_equal(velocity_indices(g, dm_v)[order], np.arange(order.size)))
P_int, _ = dm_v.coarsen().createInterpolation(dm_v)
print("interpolation coarse -> fine:", P_int.getSize()[::-1], "velocity unknowns")

same solution in DMStag ordering: 1.1266543253896089e-12
velocity block order == velocity-only DMStag order: True
interpolation coarse -> fine: (110, 412) velocity unknowns


In [6]:
def stag_mg_stokes_solve(g, eta_c, eta_n, rho_vz, rtol=1e-10, u_rtol=1e-8, max_it=300):
    """As petsc_stokes_solve, but in DMStag ordering with geometric multigrid (DMStag hierarchy,
    Galerkin coarse operators, Chebyshev/Jacobi smoothing) on the velocity block."""
    nx, nz = g.nx, g.nz
    levels = 1
    while nx % 2**levels == 0 and nz % 2**levels == 0 and min(nx, nz) // 2**levels >= 4:
        levels += 1
    dm, dm_v = make_dmstag(nx, nz, g.L, g.H), make_dmstag(nx, nz, g.L, g.H, dofs=(0, 1, 0))
    perm = ours_to_stag(g, dm)
    nv = g.n_vx + g.n_vz

    # our symmetric system, permuted into DMStag ordering
    A, s, walls = stokes_symmetric(g, eta_c, eta_n)
    b = np.zeros(A.shape[0]); bz = rho_vz.copy(); bz[0] = bz[-1] = 0.0
    b[g.n_vx:g.n_vx + g.n_vz] = bz.ravel(); b *= s; b[walls] = 0.0
    n = A.shape[0]
    Pm = sp.csr_matrix((np.ones(n), (perm, np.arange(n))), shape=(n, n))
    A = (Pm @ A @ Pm.T).tocsr(); A.sort_indices(); b = Pm @ b
    M = PETSc.Mat().createAIJ(size=A.shape, csr=(A.indptr, A.indices, A.data))

    is_u = np.sort(velocity_indices(g, dm)).astype(PETSc.IntType)
    is_p = np.sort(perm[nv:]).astype(PETSc.IntType)
    null = np.zeros(n); null[is_p] = 1 / np.sqrt(g.n_p)
    nvec = M.createVecRight(); nvec.setArray(null)
    M.setNullSpace(PETSc.NullSpace().create(vectors=[nvec]))
    eta_p = np.empty(g.n_p); eta_p[np.argsort(np.argsort(perm[nv:]))] = eta_c.ravel()   # eta in pressure-block order
    S = sp.diags(-1.0 / eta_p).tocsr()
    S_pc = PETSc.Mat().createAIJ(size=S.shape, csr=(S.indptr, S.indices, S.data))

    ksp = PETSc.KSP().create(); ksp.setOperators(M); ksp.setType("fgmres")
    pc = ksp.getPC(); pc.setType("fieldsplit")
    pc.setFieldSplitIS(("u", PETSc.IS().createGeneral(is_u)), ("p", PETSc.IS().createGeneral(is_p)))
    pc.setFieldSplitType(PETSc.PC.CompositeType.SCHUR)
    pc.setFieldSplitSchurFactType(PETSc.PC.FieldSplitSchurFactType.UPPER)
    pc.setFieldSplitSchurPreType(PETSc.PC.FieldSplitSchurPreType.USER, S_pc)
    ksp.setTolerances(rtol=rtol, max_it=max_it)
    ksp.setUp()

    ku, kp = pc.getFieldSplitSubKSP()
    kp.setType("preonly"); kp.getPC().setType("jacobi")
    ku.setType("cg"); ku.setTolerances(rtol=u_rtol)
    pmg = ku.getPC(); pmg.setOptionsPrefix("u_"); pmg.setType("mg"); pmg.setMGLevels(levels)
    hierarchy = [dm_v]
    for _ in range(levels - 1):
        hierarchy.append(hierarchy[-1].coarsen())
    hierarchy = hierarchy[::-1]                                  # coarse -> fine
    for l in range(1, levels):
        P_l, _ = hierarchy[l - 1].createInterpolation(hierarchy[l])
        pmg.setMGInterpolation(l, P_l)
    o = PETSc.Options()
    o["u_pc_mg_galerkin"] = "both"                              # coarse operators R A P from our fine matrix
    o["u_mg_levels_ksp_type"] = "chebyshev"; o["u_mg_levels_pc_type"] = "jacobi"; o["u_mg_levels_ksp_max_it"] = 3
    o["u_mg_coarse_ksp_type"] = "preonly"; o["u_mg_coarse_pc_type"] = "lu"
    pmg.setFromOptions()

    x, rhs = M.createVecRight(), M.createVecLeft(); rhs.setArray(b)
    t0 = time.time(); ksp.solve(rhs, x); elapsed = time.time() - t0
    vx, vz, P = unpack(g, x.getArray().copy()[perm])
    return vx, vz, P, ksp.getIterationNumber(), ku.getIterationNumber(), ksp.getConvergedReason(), elapsed

In [7]:
print("isoviscous (analytic):")
for n in (32, 64, 128, 256, 512):
    g = Grid(n, n)
    Xz, Zz = np.meshgrid(g.xc, g.zn); Xx, Zx = np.meshgrid(g.xn, g.zc)
    vx, vz, P, its, u_its, reason, dt = stag_mg_stokes_solve(g, np.ones((n, n)), np.ones((n + 1, n + 1)), np.cos(k * Xz) * np.sin(k * Zz))
    A0 = k / (2 * k**2) ** 2
    err = max(np.abs(vx - A0 * k * np.sin(k * Xx) * np.cos(k * Zx)).max(), np.abs(vz + A0 * k * np.cos(k * Xz) * np.sin(k * Zz)).max()) / (A0 * k)
    print(f"  {n:4d}²: error {err:.2e}, outer {its}, inner (multigrid) {u_its}, {dt:.2f} s")

print("\nstiff lid vs SciPy direct solver (128²):")
for contrast in (1e3, 1e6, 1e8):
    n = 128; g = Grid(n, n); Xz, Zz = np.meshgrid(g.xc, g.zn)
    eta_c = contrast ** np.meshgrid(g.xc, g.zc)[1]; eta_n = contrast ** np.meshgrid(g.xn, g.zn)[1]
    rho = np.cos(k * Xz) * np.sin(k * Zz)
    vx0, vz0, _ = solve_stokes(g, eta_c, eta_n, rho)
    vx, vz, P, its, u_its, reason, dt = stag_mg_stokes_solve(g, eta_c, eta_n, rho)
    diff = max(np.abs(vx - vx0).max(), np.abs(vz - vz0).max()) / np.abs(vz0).max()
    print(f"  contrast {contrast:.0e}: diff {diff:.1e}, outer {its}, inner {u_its}, {dt:.2f} s")

isoviscous (analytic):
    32²: error 8.03e-04, outer 4, inner (multigrid) 6, 0.01 s
    64²: error 2.01e-04, outer 4, inner (multigrid) 6, 0.02 s
   128²: error 5.02e-05, outer 4, inner (multigrid) 6, 0.08 s
   256²: error 1.25e-05, outer 4, inner (multigrid) 6, 0.31 s
   512²: error 3.14e-06, outer 4, inner (multigrid) 6, 1.31 s

stiff lid vs SciPy direct solver (128²):
  contrast 1e+03: diff 8.3e-09, outer 8, inner 5, 0.13 s
  contrast 1e+06: diff 3.7e-07, outer 9, inner 5, 0.15 s
  contrast 1e+08: diff 1.2e-06, outer 10, inner 5, 0.16 s
